In [1]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q -U av faster-whisper fastapi uvicorn python-multipart pyngrok soundfile requests nvidia-cublas-cu12 nvidia-cudnn-cu12

import os, glob, ctypes, subprocess, time, tempfile
import httpx
from fastapi import Request
from fastapi.responses import StreamingResponse
import requests
from google.colab import userdata

NGROK_TOKEN = "your-own-ngrok-token"
SECRET = "random-string"

# --- 1) Ollama: start, wait until ready, pull model ---
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"])
for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434"); break
    except requests.ConnectionError:
        time.sleep(1)
!ollama pull gemma4:e4b

# --- 2) preload CUDA 12 libs (BEFORE importing faster_whisper) ---
import nvidia.cublas, nvidia.cudnn
for pkg in (nvidia.cublas, nvidia.cudnn):
    libdir = os.path.join(list(pkg.__path__)[0], "lib")
    for so in sorted(glob.glob(libdir + "/lib*.so*")):
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            pass

import uvicorn
import soundfile as sf
from fastapi import FastAPI, UploadFile, Form, Header, HTTPException
from faster_whisper import WhisperModel
from pyngrok import ngrok, conf

# --- 3) ngrok: token first, then kill old tunnels, then open both ---
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
# ollama_url = ngrok.connect(11434).public_url
whisper_url = ngrok.connect(8000).public_url
print("WHISPER+OLLAMA URL:", whisper_url)

# --- 4) Whisper API (unchanged) ---
model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
app = FastAPI()

@app.api_route("/ollama/{path:path}", methods=["GET", "POST", "DELETE"])
async def ollama_proxy(path: str, request: Request, x_key: str = Header(None)):
    if x_key != SECRET:
        raise HTTPException(401)
    client = httpx.AsyncClient(timeout=None)
    req = client.build_request(
        request.method, f"http://127.0.0.1:11434/{path}",
        content=await request.body(),
        headers={"content-type": request.headers.get("content-type", "application/json")},
    )
    r = await client.send(req, stream=True)

    async def gen():
        async for chunk in r.aiter_raw():
            yield chunk
        await r.aclose(); await client.aclose()

    return StreamingResponse(gen(), status_code=r.status_code,
                             media_type=r.headers.get("content-type"))

@app.post("/transcribe")
def transcribe(file: UploadFile,
               translate: bool = Form(False),
               x_key: str = Header(None)):
    if x_key != SECRET:
        raise HTTPException(401)
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        f.write(file.file.read())
        path = f.name
    try:
        audio, sr = sf.read(path, dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        segs, _ = model.transcribe(audio, task="translate" if translate else "transcribe")
        text = " ".join(s.text.strip() for s in segs)
    finally:
        os.remove(path)
    return {"text": text}

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
await uvicorn.Server(config).serve()

In [9]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)